# BioCycle AI — Training the YOLO11 Organic Waste Detector

This notebook trains the **YOLO11m** model used in the BioCycle AI app, evaluates it, exports it to TFLite for on-device use, and shows the Gemini compost/feed advice step.

**Final results** (validation set: 3,337 images, 10,687 objects):

| mAP@50 | mAP@50-95 | Precision | Recall |
|---|---|---|---|
| 0.842 | 0.703 | 0.838 | 0.759 |

**Classes (12):** apple, banana, bread, carrot, citrus, cucumber, egg, leafy_greens, onion, pepper, potato, tomato

### Before you run
- Runtime: **GPU** (the final model was trained on an NVIDIA A100).
- Add these keys in Colab's **Secrets** panel (🔑 icon on the left). Never paste keys into the notebook:
  - `ROBOFLOW_API_KEY` — to download the dataset (the dataset is private)
  - `GEMINI_API_KEY` — only needed for the last section

In [ ]:
!nvidia-smi

In [ ]:
%pip install -q ultralytics roboflow google-genai

## 1. Setup and dataset

In [ ]:
from google.colab import drive, userdata

drive.mount('/content/drive')

# Training runs are saved to Drive so they survive runtime resets
RUNS_DIR = '/content/drive/MyDrive/yolo_models'

In [ ]:
from roboflow import Roboflow

rf = Roboflow(api_key=userdata.get('ROBOFLOW_API_KEY'))
project = rf.workspace('iot-new').project('iot-artik-last')
dataset = project.version(1).download('yolov11')

DATA_YAML = f'{dataset.location}/data.yaml'
!cat "{DATA_YAML}"

## 2. Training

Settings of the final run (`organic_v13_Medium`):

- **Model:** YOLO11m, starting from COCO-pretrained weights
- **Image size:** 640, **epochs:** up to 60 with early stopping (patience 10)
- **Batch:** the original run used `batch=144` on an A100. `batch=-1` lets Ultralytics pick the largest batch that fits the GPU, so the cell also runs on smaller GPUs.

The model went through 13 iterations before this run, from YOLOv8n on an 8-class dataset to YOLO11m on the current 12-class dataset.

In [ ]:
from ultralytics import YOLO

model = YOLO('yolo11m.pt')

results = model.train(
    data=DATA_YAML,
    epochs=60,
    imgsz=640,
    batch=-1,          # auto batch size (original run: 144 on A100)
    patience=10,       # early stopping
    project=RUNS_DIR,
    name='organic_v13_Medium',
    save_period=5,     # keep a checkpoint every 5 epochs
    plots=True,
    workers=8,
)

## 3. Evaluation

In [ ]:
RUN_DIR = f'{RUNS_DIR}/organic_v13_Medium'
best = YOLO(f'{RUN_DIR}/weights/best.pt')

metrics = best.val(data=DATA_YAML, imgsz=640)
print(f"mAP@50:    {metrics.box.map50:.3f}")
print(f"mAP@50-95: {metrics.box.map:.3f}")
print(f"Precision: {metrics.box.mp:.3f}")
print(f"Recall:    {metrics.box.mr:.3f}")

print('\nPer-class mAP@50-95:')
for i, name in best.names.items():
    print(f'  {name:<14} {metrics.box.maps[i]:.3f}')

In [ ]:
from IPython.display import Image, display

for plot in ['results.png', 'confusion_matrix_normalized.png', 'BoxPR_curve.png']:
    display(Image(filename=f'{RUN_DIR}/{plot}', width=900))

## 4. Inference on a sample image

Ultralytics letterboxes the image to 640x640 (aspect ratio kept, gray padding). The mobile app does the same before inference.

In [ ]:
import glob
from collections import Counter
import matplotlib.pyplot as plt

sample = sorted(glob.glob(f'{dataset.location}/test/images/*.jpg'))[0]
result = best.predict(sample, conf=0.25)[0]

plt.figure(figsize=(10, 10))
plt.imshow(result.plot()[..., ::-1])  # BGR -> RGB
plt.axis('off')
plt.show()

detected = [best.names[int(c)] for c in result.boxes.cls]
print(Counter(detected))

## 5. Export for the mobile app

`nms=True` builds non-maximum suppression into the model, so the app receives final boxes as `[1, 300, 6]` rows of `(x1, y1, x2, y2, confidence, class)`. `half=True` stores weights in float16, halving the file size (about 40 MB) with negligible accuracy loss.

Copy the exported `.tflite` file to `assets/models/best.tflite` in the app.

In [ ]:
tflite_path = best.export(format='tflite', imgsz=640, half=True, nms=True)
print(tflite_path)

## 6. Compost and animal-feed advice with Gemini

The detected items are sent to Gemini with a fixed expert prompt. The prompt is in Turkish because the app's users are Turkish.

In [ ]:
from google import genai

client = genai.Client(api_key=userdata.get('GEMINI_API_KEY'))

def get_compost_advice(detected_items):
    """Asks Gemini for feed/compost advice for the detected waste items."""
    if not detected_items:
        return 'No organic waste detected.'

    waste_data = ', '.join(detected_items)
    prompt = f"""
    Sen uzman bir Ziraat Mühendisi ve Veteriner Hekimsin.
    Elimde görüntü işleme ile tespit edilmiş şu gıda atıkları var: "{waste_data}".

    Lütfen aşağıdaki kurallara ve formata SIKI SIKIYA uyarak cevap ver:

    **KURALLAR:**
    1. ASLA solucan, böcek, larva veya sinek kompostu önerme.
    2. Sadece **Çiftlik Hayvanları** (İnek, Koyun, Keçi, Tavuk, Kaz) ve **Sokak Hayvanları** (Kedi, Köpek, Kuş) üzerine odaklan.
    3. Hayvanlar (kedi/köpek vs) için toksik olan gıdalarda (soğan, sarımsak, çikolata, üzüm vb.) ÇOK NET uyarı ver.

    **CEVAP FORMATI:**

    --- 🐾 HAYVAN BESLEME VE YEM RAPORU 🐾 ---

    ✅ **Uygun Olduğu Hayvanlar:**
    (Bu karışımı hangi hayvanlar güvenle yiyebilir?)

    ⛔ **Zehirli/Tehlikeli Olduğu Hayvanlar:**
    (Bu listedeki ürünlerden hangisi kime verilmez?)

    🥣 **Zenginleştirilmiş Mama/Yem Reçetesi:**
    (Bu atıkları daha besleyici yapmak için içine dışarıdan ne katalım?)

    💡 **Sonuç:**
    (Tek kelime ile: YEM veya GÜBRE veya İMHA)
    """

    response = client.models.generate_content(model='gemini-2.5-flash', contents=prompt)
    return response.text

counts = Counter(detected)
print(get_compost_advice([f'{name} x{n}' for name, n in counts.items()]))